In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [9]:
df = pd.read_csv('heart_tratado.csv', sep=',', encoding='utf-8')
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

In [10]:
categoricas = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
numericas = [col for col in X.columns if col not in categoricas]

In [ ]:
preprocess_onehot = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numericas),
        
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categoricas) 
    ])

preprocess_ordinal = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numericas),
        
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('onehot', OrdinalEncoder())
        ]), categoricas) 
    ])

In [12]:
pipeline_onehot = Pipeline([
    ('pre', preprocess_onehot),
    ('clf', XGBClassifier(max_depth=2, learning_rate=0.05, n_estimators=250, random_state=3))
])

pipeline_ordinal  = Pipeline([
    ('pre', preprocess_ordinal),
    ('clf', XGBClassifier(max_depth=2, learning_rate=0.05, n_estimators=250, random_state=3))
])

In [13]:
print("\nAvaliação com dados de teste (OneHot):")
pipeline_onehot.fit(X_train, y_train)
y_pred1 = pipeline_onehot.predict(X_test)
print(f"Acurácia {accuracy_score(y_test, y_pred1):.4f}")
print(confusion_matrix(y_test, y_pred1))
print(classification_report(y_test, y_pred1))

print("\nAvaliação com dados de teste (Ordinal):")
pipeline_ordinal.fit(X_train, y_train)
y_pred2 = pipeline_ordinal.predict(X_test)
print(f"Acurácia {accuracy_score(y_test, y_pred2):.4f}")
print(confusion_matrix(y_test, y_pred2))
print(classification_report(y_test, y_pred2))


Avaliação com dados de teste (OneHot):
Acurácia 0.8652
[[ 82  17]
 [ 14 117]]
              precision    recall  f1-score   support

           0       0.85      0.83      0.84        99
           1       0.87      0.89      0.88       131

    accuracy                           0.87       230
   macro avg       0.86      0.86      0.86       230
weighted avg       0.86      0.87      0.86       230


Avaliação com dados de teste (Ordinal):
Acurácia 0.8609
[[ 81  18]
 [ 14 117]]
              precision    recall  f1-score   support

           0       0.85      0.82      0.84        99
           1       0.87      0.89      0.88       131

    accuracy                           0.86       230
   macro avg       0.86      0.86      0.86       230
weighted avg       0.86      0.86      0.86       230



In [14]:
print("\nOneHot:")
scores1 = cross_val_score(pipeline_onehot, X, y, cv=5)
print(f"Acurácia média: {scores1.mean():.4f}")

print("\nOrdinal:")
scores2 = cross_val_score(pipeline_ordinal, X, y, cv=5)
print(f"Acurácia média: {scores2.mean():.4f}")


OneHot:
Acurácia média: 0.8331

Ordinal:
Acurácia média: 0.8364
